# Module 5.2: One Agent, Many Memories

Notebook 5.1 gave the agent a safe, bounded way to recall Sarah's **preferences**. But Sarah's requests rarely stay inside one kind of memory:

> **"Book the same hotel I stayed at in London last time, as long as it fits my hotel budget."**

Answering this needs *what happened* (episodic: the last London trip), *what is allowed* (policy: the Senior hotel limit) and *what is available* (operational: hotel search). Preferences alone cannot answer it.

**How does one agent decide which memory to consult for each request?**

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

import certifi
import sniffio

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.cosmos.aio import CosmosClient
from azure.identity.aio import (
    AzureCliCredential as AsyncCliCredential,
    get_bearer_token_provider as async_get_bearer_token_provider,
)
from azure.search.documents import SearchClient
from openai import AsyncAzureOpenAI
from agent_framework import (
    Agent, AgentSession, FunctionInvocationContext, SkillsProvider,
    ToolApprovalMiddleware, function_middleware,
)
from shared.memory_tools import make_episodic_tool, make_policy_tool, make_retrieval_tool
from shared.semantic_store import SemanticMemoryStore, create_container
from shared.travel_agent import SYSTEM_PROMPT, create_client, search_flights, search_hotels

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Foundry client ready")

In [ ]:
embed_token = async_get_bearer_token_provider(
    AsyncCliCredential(), "https://cognitiveservices.azure.com/.default"
)
embed_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
embed_client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=embed_token,
    api_version="2024-02-01",
)

async def embed(text: str) -> list[float]:
    r = await embed_client.embeddings.create(input=[text], model=embed_deployment)
    return r.data[0].embedding

SCOPE = "nb05_routing"
cosmos = CosmosClient(os.environ["COSMOS_ENDPOINT"], credential=AsyncCliCredential())
semantic_container = await create_container(cosmos)
episodic_container = cosmos.get_database_client(
    os.environ.get("COSMOS_DATABASE", "travel-memory")
).get_container_client("episodic-events")
search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name="travel-policies",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"]),
)
# Resolve the path: skill resources reached via '..' are rejected as outside the skill folder.
skills = SkillsProvider.from_paths(str(Path("../02_memory_layers/skills").resolve()))
print("Cosmos DB, AI Search and skills ready")

## Seeding Sarah's Memories

A small, scoped set for this notebook: a few preferences in semantic memory and two past trips in episodic memory. The `scope_tag` keeps them apart from the earlier notebooks' data and lets us clean up at the end.

In [ ]:
sarah = SemanticMemoryStore(semantic_container, user_id="E001", embed_fn=embed, scope_tag=SCOPE)
await sarah.reset()
for category, text, confidence, state in [
    ("hotel_chain", "Prefers to stay at Marriott hotels", 0.95, "trusted"),
    ("meal", "Orders vegetarian meals on flights", 0.90, "trusted"),
    ("seating", "Prefers aisle seats on flights", 0.70, "provisional"),
]:
    await sarah.add_preference(category, text, confidence, "user_assertion", state=state)

EVENTS = [
    ("trip", "Trip to London", {"city": "London", "hotel": "London Marriott Hotel Park Lane",
                                "dates": "2026-03-09 to 2026-03-13"}, "2026-03-13T18:00:00+00:00"),
    ("feedback", "Rated London hotel 5/5", {"hotel": "London Marriott Hotel Park Lane",
                                            "comment": "Quiet room, great breakfast"}, "2026-03-14T09:00:00+00:00"),
    ("trip", "Trip to Chicago", {"city": "Chicago", "hotel": "Hilton Chicago O'Hare",
                                 "dates": "2026-09-14 to 2026-09-16", "issue": "Inbound flight delayed 3 hours"},
     "2026-09-16T20:00:00+00:00"),
]
event_ids = []
for n, (event_type, description, details, ts) in enumerate(EVENTS):
    doc = {"id": f"{SCOPE}-E001-{n}", "user_id": "E001", "event_type": event_type,
           "description": description, "details": details, "timestamp": ts, "scope_tag": SCOPE}
    await episodic_container.upsert_item(doc)
    event_ids.append(doc["id"])
print(f"Seeded {await sarah.count()} preferences and {len(event_ids)} episodic events for E001")

## Tracing Tool Calls

To see routing decisions we record every tool the agent invokes with MAF function middleware. `ToolApprovalMiddleware` auto-approves the skill tools, as in Module 2.4.

In [ ]:
calls: list[tuple[str, str]] = []

@function_middleware
async def trace(context: FunctionInvocationContext, call_next):
    await call_next()
    calls.append((context.function.name, str(context.result)))

approval = ToolApprovalMiddleware(auto_approval_rules=[SkillsProvider.all_tools_auto_approval_rule])

async def ask(agent, question, *, session=None, expect=()):
    calls.clear()
    result = await agent.run(question, session=session or AgentSession())
    names = [name for name, _ in calls]
    print(f"USER: {question}\nTOOLS: {names or '(none)'}\nASSISTANT: {result.text}\n")
    missing = set(expect) - set(names)
    assert not missing, f"expected tool(s) not called: {missing}"
    return names

## The Problem: One Memory Surface

The 5.1 agent has a single persistent-memory tool, `recall_current_beliefs`, plus hotel search. Sarah asks the London question.

In [ ]:
QUESTION = "Book the same hotel I stayed at in London last time, as long as it fits my hotel budget."

semantic_only = Agent(
    client=client,
    name="SemanticOnlyAssistant",
    instructions=SYSTEM_PROMPT + "\n\nThe current user is Sarah Chen (E001), a Senior employee.",
    tools=[make_retrieval_tool(sarah), search_hotels],
    middleware=[trace],
)
await ask(semantic_only, QUESTION)

## What Went Wrong

| The request needs | Where it lives | Reachable by the 5.1 agent? |
|---|---|---|
| Which hotel Sarah stayed at in London | Episodic memory (`episodic-events`) | No |
| Her hotel limit as a Senior employee | Corporate policy (Azure AI Search) | No |
| How to book international travel | Procedural memory (skills) | No |
| Which chain she prefers | Semantic memory | Yes |
| What is available | Hotel search | Yes |

At best the agent substitutes the preference ("Marriott") for the fact ("this specific Marriott") and guesses at the budget. Each memory type answers a different question; an agent that can reach only one of them answers the others badly.

## The Solution: Every Surface as a Well-Described Tool

```mermaid
flowchart LR
    U[User request] --> L[LLM tool selection]
    L --> S[AgentSession: this conversation]
    L --> E[recall_past_events: what happened]
    L --> M[recall_current_beliefs: what is true about Sarah]
    L --> P[SkillsProvider: how to do it]
    L --> R[search_travel_policies: what is allowed]
    L --> O[search_flights / search_hotels: what is available]
```

There is no router class. The model already selects tools by reading their descriptions, so **the descriptions are the routing table**. Every personal-memory tool is built for one user (`make_*_tool(store_or_container, user_id)`): the model chooses *which* memory to read, never *whose*.

In [ ]:
def build_agent(store: SemanticMemoryStore, user_id: str) -> Agent:
    return Agent(
        client=client,
        name="RoutingTravelAssistant",
        instructions=SYSTEM_PROMPT + "\n\n" + (
            "Choose tools by what the request needs; combine them for compound requests. "
            "Answer questions about this conversation from the conversation itself. "
            "Present recalled preferences as their presentation says (assert as fact, hedge "
            "and confirm, or ask) in natural language; never echo raw tool output. "
            "Before booking, load the matching booking skill and follow it. "
            f"The current user is employee {user_id}, a Senior employee."
        ),
        tools=[
            make_retrieval_tool(store),
            make_episodic_tool(episodic_container, user_id, scope_tag=SCOPE),
            make_policy_tool(search_client, embed),
            search_flights,
            search_hotels,
        ],
        context_providers=[skills],
        middleware=[approval, trace],
    )

assistant = build_agent(sarah, "E001")
print(f"Agent ready: {assistant.name}")

### Single-surface questions

Each question must reach the surface that knows the answer. The model may add context from another surface (for example, preferences while explaining a booking procedure); the check is that the right one is never skipped.

In [ ]:
await ask(assistant, "Which hotel chain do I prefer?", expect=["recall_current_beliefs"])

In [ ]:
await ask(assistant, "What happened on my last trip to Chicago?", expect=["recall_past_events"])

In [ ]:
await ask(assistant, "What steps must I follow to book an international trip?", expect=["load_skill"])

In [ ]:
await ask(assistant, "What is the nightly hotel limit for a Senior employee?", expect=["search_travel_policies"])

### The compound request

The original question, now with every surface available. The agent should combine episodic memory (which hotel), policy (the limit) and hotel search.

In [ ]:
await ask(assistant, QUESTION, expect=["recall_past_events", "search_travel_policies", "search_hotels"])

### Session memory needs no tool

A question about the current conversation is answered from `AgentSession` history. No persistent-memory tool should run.

In [ ]:
session = AgentSession()
await ask(assistant, "Which hotel chain do I prefer?", session=session)
names = await ask(assistant, "What did I just ask you?", session=session)
persistent = {"recall_current_beliefs", "recall_past_events", "search_travel_policies"}
assert not persistent & set(names), f"session question used persistent memory: {names}"
print("Session question answered from conversation history only")

## User Isolation

The same agent definition, built for a different employee. Routing works exactly as before, but every personal-memory tool reads only E002's partition, so none of Sarah's memories can appear in a tool result.

In [ ]:
other_store = SemanticMemoryStore(semantic_container, user_id="E002", embed_fn=embed, scope_tag=SCOPE)
other_assistant = build_agent(other_store, "E002")
await ask(other_assistant, "Which hotel chain do I prefer, and where did I stay on my last London trip?")

sarah_texts = ["Marriott", "Park Lane", "vegetarian", "aisle"]
leaks = [(name, t) for name, result in calls if name in ("recall_current_beliefs", "recall_past_events")
         for t in sarah_texts if t in result]
assert not leaks, f"cross-user leak: {leaks}"
print("No E001 memory reached E002's tool results")

## Key Takeaways

1. **Each memory type answers a different question.** Session: *this conversation*. Episodic: *what happened*. Semantic: *what is true about the user*. Procedural: *how to do it*. Policy: *what is allowed*.
2. **The LLM is the router.** Clear tool descriptions replace a hand-written classifier, and compound requests combine tools naturally.
3. **Descriptions are the routing table.** "Policy applies to everyone; it is not a personal preference" keeps the model from treating a budget rule as Sarah's choice.
4. **Session questions stay in session.** Conversation history answers "what did I just ask?" without touching persistent memory.
5. **Tools are bound to one user.** `make_retrieval_tool(store)` and `make_episodic_tool(container, user_id)` fix the partition key at construction; the model never supplies it.
6. **Semantic recall keeps 5.1's policy.** `recall_current_beliefs` still applies current-only, trust gating, ranking and the `top_k` bound.

## Next: Governance

The agent can now reach every memory it needs. Module 6 gives Sarah control over those memories: inspect, correct, retire and erase.

In [ ]:
print(f"Removed {await sarah.reset()} preferences")
for doc_id in event_ids:
    await episodic_container.delete_item(doc_id, partition_key="E001")
print(f"Removed {len(event_ids)} episodic events")
await cosmos.close()